In [1]:
# Import all required libraries
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import os
import platform
import datetime as dt

In [2]:
os.makedirs('data', exist_ok=True)
print("python", platform.python_version())
print("pandas", pd.__version__)
print("numpy", np.__version__)
print("yfinance", yf.__version__)

python 3.13.18
pandas 2.2.3
numpy 2.4.6
yfinance 0.2.66


In [3]:
# Part 1: Extract S&P 500 tickers from Wikipedia using Pandas
# User-Agent is required to avoid HTTP 403 blocking from Wikipedia
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
table = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
sp500_df = table[0]

tickers = sp500_df['Symbol'].tolist()
tickers = [t.replace('.', '-') for t in tickers]

retrieval_date = dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%d %H:%M UTC')
sp500_df['Ticker'] = tickers
sp500_df['Retrieval_Date'] = retrieval_date
sp500_df.to_csv('data/SP500_tickers.csv', index=False)
print('Retrieved:', retrieval_date)

print('Total tickers extracted:', len(tickers))
sp500_df[['Symbol', 'Security', 'GICS Sector']].head(10)

Retrieved: 2026-11-20 03:19 UTC
Total tickers extracted: 503


,Symbol,Security,GICS Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology
5,ADBE,Adobe Inc.,Information Technology
6,AMD,Advanced Micro Devices,Information Technology
7,AES,AES Corporation,Utilities
8,AFL,Aflac,Financials
9,A,Agilent Technologies,Health Care


In [4]:
#Part2: Load Daily Market Data via yfinance
# Download daily closing prices for all S&P 500 tickers
# Period: 01-01-2021 to 01-01-2025 | Close price (adjusted)
start = '2021-01-01'
end = '2025-01-01'

data = yf.download(tickers, start=start, end=end, auto_adjust=False, actions = True)
df = data['Adj Close']
close = data['Close']
volume = data['Volume']

df.to_csv('data/SP500_prices.csv')
close.to_csv('data/SP500_close.csv')
volume.to_csv('data/SP500_volume.csv')
print('Shape:', df.shape)
df.head()



[*********************100%***********************]  503 of 503 completed
ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['HONA', 'Q', 'SNDK', 'VYLR', 'FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Shape: (1005, 503)


Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-04,114.009109,125.537125,84.219154,139.149994,97.269707,33.186359,232.029343,485.339996,132.622620,42.189487,...,26.124495,102.852097,54.757393,33.042759,93.015907,221.160004,95.016190,141.603348,378.130005,152.952011
2021-01-05,114.941269,127.089241,85.090012,148.300003,98.473206,33.319485,233.350296,485.690002,134.993179,43.010098,...,26.371326,106.017517,54.223652,34.635185,92.903938,229.750000,95.025169,144.053421,380.570007,154.017883
2021-01-06,118.093231,122.811226,84.354973,142.770004,98.268158,34.783871,235.901611,466.309998,135.601669,44.752842,...,26.506689,106.652542,55.424541,35.518978,96.822739,226.889999,95.545944,148.158386,394.820007,156.280518
2021-01-07,121.235603,127.001915,85.257774,151.270004,99.222061,34.460564,238.127319,477.739990,141.332703,44.355221,...,26.761484,105.594177,54.340405,35.797653,101.217415,239.479996,94.818626,147.280029,409.100006,156.140289
2021-01-08,122.100464,128.098083,85.705215,149.770004,99.498413,34.650742,238.995834,485.100006,142.359055,44.473660,...,26.578350,104.901451,54.840775,36.195755,100.191048,241.449997,96.210396,146.956482,405.470001,157.178085


In [5]:
print('Duplicate dates:', df.index.duplicated().sum())
print('Non-positive prices:', list(df.columns[((df <= 0) | (close <= 0)).any()]))
print('Non-positive volumes:', list(df.columns[((volume <= 0)).any()]))
print('Zero volume days:', list(df.columns[((volume == 0) & close.notna()).any()]))

if 'Stock Splits' in data.columns.get_level_values(0):
  splits = data['Stock Splits'].fillna(0)
  splits_check = [(d.date(), t, splits.at[d, t], close[t].pct_change(fill_method=None).at[d])
        for t in splits.columns
        for d in splits.index[splits[t] > 0]]
  print(pd.DataFrame(splits_check, columns=['Date','Share','Ratio','Close_return_on_day']))

Duplicate dates: 0
Non-positive prices: []
Non-positive volumes: ['FERG', 'SW']
Zero volume days: ['FERG', 'SW']
          Date Share  Ratio  Close_return_on_day
0   2022-06-06  AMZN  20.00             0.019943
1   2021-11-18  ANET   4.00             0.003558
2   2024-12-04  ANET   4.00             0.017517
3   2021-03-05   APH   2.00             0.025316
4   2024-06-12   APH   2.00             0.021945
..         ...   ...    ...                  ...
56  2024-02-26   WMT   3.00             0.018455
57  2022-03-24   WRB   1.50             0.000613
58  2024-07-11   WRB   1.50            -0.015007
59  2024-07-09   WSM   2.00            -0.002595
60  2022-03-01   ZBH   1.03             0.005301

[61 rows x 4 columns]


In [6]:
# Transpose DataFrame: rows = tickers, columns = dates
df1 = df.T

print('Shape after transpose:', df1.shape)
df1


Shape after transpose: (503, 1005)


Date,2021-01-04,2021-01-05,2021-01-06,2021-01-07,2021-01-08,2021-01-11,2021-01-12,2021-01-13,2021-01-14,2021-01-15,...,2024-12-17,2024-12-18,2024-12-19,2024-12-20,2024-12-23,2024-12-24,2024-12-26,2024-12-27,2024-12-30,2024-12-31
Ticker,,,,,,,,,,,,,,,,,,,,,
A,114.009109,114.941269,118.093231,121.235603,122.100464,123.147919,122.013992,122.023575,121.110680,121.283661,...,134.353821,130.957016,131.134277,132.433884,132.522507,133.999374,133.733521,133.447983,132.345276,132.510986
AAPL,125.537125,127.089241,122.811226,127.001915,128.098083,125.119972,124.945374,126.972824,125.052086,123.335030,...,251.462692,246.075882,247.802048,252.464630,253.238449,256.145142,256.958557,253.555893,250.192856,248.427032
ABBV,84.219154,85.090012,84.354973,85.257774,85.705215,87.103424,87.710625,89.843864,90.231857,89.334625,...,164.530701,162.316696,160.909500,164.718369,167.354507,168.864914,168.114395,166.998001,165.299973,166.707184
ABNB,139.149994,148.300003,142.770004,151.270004,149.770004,148.130005,160.800003,169.990005,180.500000,169.270004,...,132.240005,127.540001,129.350006,134.210007,134.520004,134.990005,135.320007,133.384995,131.809998,131.410004
ABT,97.269707,98.473206,98.268158,99.222061,99.498413,98.811966,97.029007,99.979797,98.493980,99.621788,...,108.486847,107.184517,107.653725,109.387001,109.463600,109.894531,110.382896,110.114784,108.017624,108.314484
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XYZ,221.160004,229.750000,226.889999,239.479996,241.449997,225.500000,227.520004,230.919998,232.789993,227.750000,...,94.550003,87.300003,87.639999,89.650002,89.290001,91.080002,91.480003,88.970001,87.480003,84.989998
YUM,95.016190,95.025169,95.545944,94.818626,96.210396,96.677315,96.677315,96.210396,96.776077,95.734528,...,130.267365,127.105705,126.042152,127.975906,129.426208,130.528442,131.495316,130.750839,129.097458,129.716263
ZBH,141.603348,144.053421,148.158386,147.280029,146.956482,147.002670,147.770081,150.654678,150.349548,148.038239,...,105.912392,103.756119,103.707108,104.991066,104.697037,105.167488,105.118477,104.746040,103.537766,103.763702


In [7]:
# Part 3: Identify and Remove Invalid Data
# 1. Drop tickers (rows) that are entirely NaN
# 2. Drop tickers with less than 80% of dates available
# 3. Forward-fill then back-fill remaining gaps
# 4. Drop any remaining fully NaN rows
print('Before cleaning:', df1.shape)

df1 = df1.dropna(how='all')

bad = (df1.isna().any(axis=1)
        | close.T.reindex(df1.index).isna().any(axis = 1)
        | volume.T.reindex(df1.index).isna().any(axis = 1)
        | (df1 <= 0).any(axis = 1)
        | (volume.T.reindex(df1.index) < 0).any(axis = 1))


df1 = df1[~bad]

excluded = sorted(set(tickers) - set(df1.index))
exclusion_log = pd.DataFrame({'Share': excluded,
                              'First_date': [df[t].first_valid_index() if t in df else None for t in excluded],
                              'Missing_days': [int(df[t].isna().sum()) if t in df else None for t in excluded]})
exclusion_log.to_csv('data/exclusion_log.csv', index=False)

print('After cleaning:', df1.shape)
print('Rows (tickers) removed:', 503 - df1.shape[0])
df1.head()

Before cleaning: (503, 1005)
After cleaning: (488, 1005)
Rows (tickers) removed: 15


Date,2021-01-04,2021-01-05,2021-01-06,2021-01-07,2021-01-08,2021-01-11,2021-01-12,2021-01-13,2021-01-14,2021-01-15,...,2024-12-17,2024-12-18,2024-12-19,2024-12-20,2024-12-23,2024-12-24,2024-12-26,2024-12-27,2024-12-30,2024-12-31
Ticker,,,,,,,,,,,,,,,,,,,,,
A,114.009109,114.941269,118.093231,121.235603,122.100464,123.147919,122.013992,122.023575,121.110680,121.283661,...,134.353821,130.957016,131.134277,132.433884,132.522507,133.999374,133.733521,133.447983,132.345276,132.510986
AAPL,125.537125,127.089241,122.811226,127.001915,128.098083,125.119972,124.945374,126.972824,125.052086,123.335030,...,251.462692,246.075882,247.802048,252.464630,253.238449,256.145142,256.958557,253.555893,250.192856,248.427032
ABBV,84.219154,85.090012,84.354973,85.257774,85.705215,87.103424,87.710625,89.843864,90.231857,89.334625,...,164.530701,162.316696,160.909500,164.718369,167.354507,168.864914,168.114395,166.998001,165.299973,166.707184
ABNB,139.149994,148.300003,142.770004,151.270004,149.770004,148.130005,160.800003,169.990005,180.500000,169.270004,...,132.240005,127.540001,129.350006,134.210007,134.520004,134.990005,135.320007,133.384995,131.809998,131.410004
ABT,97.269707,98.473206,98.268158,99.222061,99.498413,98.811966,97.029007,99.979797,98.493980,99.621788,...,108.486847,107.184517,107.653725,109.387001,109.463600,109.894531,110.382896,110.114784,108.017624,108.314484


In [8]:
import pandas as pd

def calculate_weekly_returns(df1, shares=None):
    """Calculate weekly returns from df1 (tickers as rows, dates as columns)."""
    prices = df1 if shares is None else df1.loc[shares]

    prices = prices.T.copy()
    prices.index = pd.to_datetime(prices.index)
    prices = prices.sort_index()

    weekly_prices = prices.resample("W-FRI").last()
    weekly_prices = weekly_prices[(weekly_prices.index - pd.Timedelta(days=6) >= start) & (weekly_prices.index < end)]
    return weekly_prices.pct_change(fill_method=None)

In [9]:
# All shares weekly returns
weekly_returns = calculate_weekly_returns(df1).iloc[1:]

In [10]:
weekly_returns


Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-15,-0.006690,-0.037183,0.042348,0.130200,0.001240,-0.048024,-0.036505,-0.055700,0.006444,-0.007038,...,-0.025764,-0.025681,-0.012166,0.053454,-0.036133,-0.056741,-0.004946,0.007361,0.009249,-0.038962
2021-01-22,0.000951,0.093834,0.003076,0.074437,0.013836,-0.039781,0.004415,0.031348,-0.013756,0.011877,...,0.021218,0.011485,-0.003079,-0.009605,-0.008696,-0.021383,0.004408,0.005558,-0.000929,0.012522
2021-01-29,-0.048761,-0.051125,-0.075591,0.009677,0.095268,-0.057040,-0.050438,-0.028935,-0.053028,-0.053199,...,-0.060825,-0.073709,-0.011890,-0.054607,-0.058577,-0.031048,-0.052292,-0.045587,-0.051389,-0.055593
2021-02-05,0.025048,0.037923,0.060988,0.063606,0.003560,0.076090,0.048032,0.072694,0.009706,0.065787,...,0.102918,0.175425,-0.008595,0.113961,0.016565,0.113076,0.037935,0.040476,0.049713,0.032609
2021-02-12,0.038967,-0.010164,-0.039455,0.088936,0.033863,0.039941,0.015500,0.013655,0.080801,0.052711,...,0.018604,0.007351,-0.029949,0.028587,0.009471,0.134662,0.006033,0.009319,0.173491,0.046648
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-29,0.030858,0.032453,0.033795,-0.008956,0.008577,0.010332,0.010344,0.007381,0.016124,0.027668,...,0.039074,0.033848,0.016959,-0.031448,0.001851,-0.040212,0.027131,0.017057,0.024699,-0.009663
2024-12-06,0.018917,0.023217,-0.036845,0.005951,-0.026690,-0.028693,0.000911,0.071773,0.000688,-0.058242,...,-0.030068,-0.000106,-0.038864,-0.037216,-0.002998,0.109543,-0.004628,-0.036307,0.005700,0.006904
2024-12-13,-0.011239,0.021784,-0.016005,-0.043602,-0.019204,-0.045078,-0.011084,-0.157823,-0.003084,0.019059,...,-0.039310,0.001695,-0.023229,-0.024038,-0.045660,-0.063817,-0.004578,-0.011941,-0.013999,0.009747


In [11]:
mean_weekly_return = weekly_returns.mean().rename('Mean_Weekly_Return')
print('Weekly observations:', len(weekly_returns))
print('Mean weekly return:', mean_weekly_return)
print(mean_weekly_return.describe())

Weekly observations: 207
Mean weekly return: Ticker
A       0.001121
AAPL    0.003930
ABBV    0.003749
ABNB    0.001475
ABT     0.000971
          ...   
XYZ    -0.001554
YUM     0.001807
ZBH    -0.001105
ZBRA    0.001181
ZTS     0.000763
Name: Mean_Weekly_Return, Length: 488, dtype: float64
count    488.000000
mean       0.002857
std        0.002646
min       -0.004468
25%        0.001182
50%        0.002629
75%        0.004207
max        0.017772
Name: Mean_Weekly_Return, dtype: float64


In [12]:
import pandas as pd
import numpy as np

def calculate_weekly_beta(weekly_returns, market_returns, min_weeks=52):
    """
    weekly_returns: DataFrame with weeks as rows and shares as columns.
    market_returns: Series of benchmark returns indexed by week.
    """
    if min_weeks < 2:
        raise ValueError("min_weeks must be at least 2.")

    results = []

    for share in weekly_returns.columns:
        paired = pd.concat(
            [weekly_returns[share].rename("share"),
             market_returns.rename("market")],
            axis=1
        ).replace([np.inf, -np.inf], np.nan).dropna()

        variance = paired["market"].var(ddof=1)
        beta = np.nan

        if len(paired) >= min_weeks and variance > 0:
            beta = paired["share"].cov(paired["market"]) / variance

        results.append({
            "Share": share,
            "Beta": beta,
            "Paired_weeks": len(paired)
        })

    return pd.DataFrame(
        results, columns=["Share", "Beta", "Paired_weeks"]
    ).set_index("Share")

In [13]:
# Pass market_weekly_returns as a Series by selecting its column
market_prices = yf.download("^GSPC", start=start, end=end, auto_adjust=False)["Adj Close"]
market_weekly_returns = calculate_weekly_returns(market_prices.T).iloc[1:]
market_weekly_returns = pd.DataFrame({"^GSPC": market_weekly_returns.squeeze()})

beta_df = calculate_weekly_beta(
    weekly_returns,
    market_weekly_returns["^GSPC"],
    min_weeks = len(market_weekly_returns)
)
print('Aligned weekly observations per beta:', beta_df['Paired_weeks'].unique())
print('Shares with missing beta:', int(beta_df['Beta'].isna().sum()))
beta_df.head()

[*********************100%***********************]  1 of 1 completed


Aligned weekly observations per beta: [207]
Shares with missing beta: 0


,Beta,Paired_weeks
Share,,
A,1.074724,207
AAPL,1.174835,207
ABBV,0.425365,207
ABNB,1.472709,207
ABT,0.583543,207


In [14]:
# Combine into one DataFrame
combined_df = (
    weekly_returns
    .rename_axis(index="Week", columns="Share")
    .reset_index()
    .melt(
        id_vars="Week",
        var_name="Share",
        value_name="Weekly_Return"
    )
    .join(beta_df[["Beta"]], on="Share")
    .sort_values(["Share", "Week"])
    .reset_index(drop=True)
)

combined_df

,Week,Share,Weekly_Return,Beta
0,2021-01-15,A,-0.006690,1.074724
1,2021-01-22,A,0.000951,1.074724
2,2021-01-29,A,-0.048761,1.074724
3,2021-02-05,A,0.025048,1.074724
4,2021-02-12,A,0.038967,1.074724
...,...,...,...,...
101011,2024-11-29,ZTS,-0.009663,0.979783
101012,2024-12-06,ZTS,0.006904,0.979783
101013,2024-12-13,ZTS,0.009747,0.979783
101014,2024-12-20,ZTS,-0.074868,0.979783


In [15]:
# Count null values in each column
print("Null values per column:")
print(combined_df.isna().sum())

# Remove rows with any null values
rows_before = len(combined_df)
combined_df = combined_df.dropna().reset_index(drop=True)

print("Rows removed:", rows_before - len(combined_df))
print("Rows remaining:", len(combined_df))

Null values per column:
Week             0
Share            0
Weekly_Return    0
Beta             0
dtype: int64
Rows removed: 0
Rows remaining: 101016


#Part 4: Stock-market RFM
R = number of calendar days between most recent high volume day and date of observation (smaller = more recent, absent = missing, not 0) F = number of high volume transactions in last 60 trading days M = sum of Close multiplied by Volume for last 60 trading days in USD (approx. dollar value traded, not market capitalisation) A transaction is defined as high volume if the volume is more than twice the average volume of the 20 days prior (excluding the day in question). Date of observation is the most recent completed trading day on which all data is common. These are not conventional customer RFM values but rather are defined for use in this research.

In [16]:
import numpy as np
import pandas as pd

vol = volume[df1.index].replace([np.inf, -np.inf], np.nan).sort_index()
vol = vol.loc[(vol.index >= '2021-01-01') & (vol.index < '2025-01-01')]
vol = vol.dropna(axis=1, how='all')
cls = close[vol.columns].loc[vol.index]

snapshot = vol.dropna().index.max()
vol, cls = vol.loc[:snapshot], cls.loc[:snapshot]

average_volume = vol.shift(1).rolling(20).mean()
high_volume = (vol > 2 * average_volume) & average_volume.notna()

recency = pd.Series({
    ticker: (snapshot - high_volume.index[np.flatnonzero(high_volume[ticker].to_numpy())[-1]]).days
            if high_volume[ticker].any() else np.nan
    for ticker in vol.columns
}, name='Recency')

last60 = vol.index[-60:]
frequency = high_volume.loc[last60].sum(axis=0).rename('Frequency')
valid_days = average_volume.notna().sum(axis=0).rename('Valid_Days')
monetary = (cls.loc[last60] * vol.loc[last60]).sum(axis=0).rename('Monetary')

rfm = pd.concat([recency, frequency, valid_days, monetary], axis=1)
print('Shares with fewer than 60 valid days:', int((rfm['Valid_Days'] < 60).sum()))

def score_five(series, ascending_better=True):
    percentile = series.rank(pct=True, method='average', ascending=ascending_better)
    return np.ceil(percentile * 5).clip(1, 5).astype('Int64')

rfm['R_Score'] = score_five(rfm['Recency'], ascending_better=False)
rfm['F_Score'] = score_five(rfm['Frequency'], ascending_better=True)
rfm['M_Score'] = score_five(rfm['Monetary'], ascending_better=True)
rfm['RFM_Score'] = rfm[['R_Score', 'F_Score', 'M_Score']].sum(axis=1)
rfm['RFM_Code'] = (rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str))
rfm = rfm.sort_values(['RFM_Score', 'Monetary', 'Frequency'], ascending=[False, False, False])
rfm['Rank'] = np.arange(1, len(rfm) + 1)
rfm.index.name = 'Share'
rfm = rfm.reset_index()

obs_date = snapshot
shares = rfm['Share'].tolist()

print('Snapshot:', snapshot.date())
print('Shares ranked:', len(rfm))
print('No high-volume event (R missing):', int(rfm['Recency'].isna().sum()))
print(rfm.head(20).to_string(index=False))


Shares with fewer than 60 valid days: 0
Snapshot: 2024-12-31
Shares ranked: 488
No high-volume event (R missing): 0
Share  Recency  Frequency  Valid_Days     Monetary  R_Score  F_Score  M_Score  RFM_Score RFM_Code  Rank
 AMZN       11          5         985 4.681770e+11        4        5        5         14      455     1
 AVGO       11          6         985 3.340388e+11        4        5        5         14      455     2
 PLTR       11          6         985 2.869115e+11        4        5        5         14      455     3
 GOOG       11          5         985 2.028245e+11        4        5        5         14      455     4
   BA       11          5         985 1.276863e+11        4        5        5         14      455     5
  MCD       11          5         985 6.310161e+10        4        5        5         14      455     6
 WDAY       11          5         985 5.170406e+10        4        5        5         14      455     7
  AJG        4          6         985 2.824304e+10  

In [17]:
ex_t = 'AAPL' if 'AAPL' in shares else shares[0]
ex = pd.DataFrame({"Close": close[ex_t], "Volume": volume[ex_t]}).loc[:obs_date]
ex["Avg20_prev"] = ex["Volume"].shift(1).rolling(20).mean()
ex["Threshold"] = 2 * ex["Avg20_prev"]
ex["HighVolume"] = ex["Volume"] > ex["Threshold"]
ex["Close_x_Volume"] = ex["Close"] * ex["Volume"]
last60 = ex.tail(60)
row = rfm.set_index("Share").loc[ex_t]

print(ex_t, '| observation date', obs_date.date())
print('High-volume days in last 60 sessions:', list(last60.index[last60["HighVolume"]].date))
print('R =', row.Recency, 'calendar days | F =', row.Frequency, '| M = $', f"{row.Monetary:,.0f}")
last60[last60["HighVolume"]].round(2) if last60["HighVolume"].any() else last60.tail(5).round(2)

AAPL | observation date 2024-12-31
High-volume days in last 60 sessions: [datetime.date(2024, 11, 25), datetime.date(2024, 12, 20)]
R = 11 calendar days | F = 2 | M = $ 623,303,377,707


,Close,Volume,Avg20_prev,Threshold,HighVolume,Close_x_Volume
Date,,,,,,
2024-11-25,232.87,90152800,43820740.0,87641480.0,True,2.099388e+10
2024-12-20,254.49,147495300,45004605.0,90009210.0,True,3.753608e+10


In [18]:
# Full RFM table and exports
from IPython.display import display

sectors = sp500_df.set_index('Ticker')['GICS Sector']
rfm = (rfm.set_index('Share')
          .join(mean_weekly_return)
          .join(beta_df[['Beta']])
          .join(sectors.rename('Sector'))
          .reset_index())

display(rfm)
display(rfm[['Mean_Weekly_Return', 'Beta', 'Recency', 'Frequency', 'Monetary']].describe())
rfm.to_csv('SP500_RFM_2021_2024.csv', index=False)
rfm.to_excel('SP500_RFM_2021_2024.xlsx', index=False)
print('Saved SP500_RFM_2021_2024.csv and SP500_RFM_2021_2024.xlsx')

,Share,Recency,Frequency,Valid_Days,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Code,Rank,Mean_Weekly_Return,Beta,Sector
0,AMZN,11,5,985,4.681770e+11,4,5,5,14,455,1,0.002681,1.386667,Consumer Discretionary
1,AVGO,11,6,985,3.340388e+11,4,5,5,14,455,2,0.010133,1.515255,Information Technology
2,PLTR,11,6,985,2.869115e+11,4,5,5,14,455,3,0.009613,2.003194,Information Technology
3,GOOG,11,5,985,2.028245e+11,4,5,5,14,455,4,0.004629,1.209198,Communication Services
4,BA,11,5,985,1.276863e+11,4,5,5,14,455,5,0.000614,1.134751,Industrials
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,JKHY,46,2,985,6.090703e+09,1,2,1,4,121,484,0.000966,0.621046,Financials
484,LNT,60,2,985,5.708137e+09,1,2,1,4,121,485,0.001908,0.492376,Utilities
485,MKC,91,0,985,7.850361e+09,1,1,1,3,111,486,-0.000156,0.449061,Consumer Staples
486,MOS,49,1,985,6.605748e+09,1,1,1,3,111,487,0.001363,0.930058,Materials


,Mean_Weekly_Return,Beta,Recency,Frequency,Monetary
count,488.000000,488.000000,488.000000,488.000000,4.880000e+02
mean,0.002857,0.973868,20.389344,2.987705,3.930290e+10
std,0.002646,0.406550,21.302843,1.328820,1.288833e+11
min,-0.004468,0.055452,4.000000,0.000000,1.297483e+09
25%,0.001182,0.666634,11.000000,2.000000,1.023194e+10
50%,0.002629,0.950610,11.000000,3.000000,1.668382e+10
75%,0.004207,1.179290,15.000000,4.000000,3.210048e+10
max,0.017772,3.778560,222.000000,8.000000,1.847697e+12


Saved SP500_RFM_2021_2024.csv and SP500_RFM_2021_2024.xlsx


### Interpretation and limitations

- Index composition on the date of data retrieval from the Wikipedia list of S&P 500 constituents differs from historical index composition. This can result in survivorship bias (companies that have been removed from the index are not included, and companies that entered the index after the start date are not included due to a lack of historical data).

- Missing prices are not filled. Tickers with gaps are dropped and can be found in the exclusion_log file.

- ^GSPC is a price index that excludes dividends, but share returns use Adj Close. The impact of weekly dividends is relatively small, but the price index is not the same as a total-returns index.

- RFM represents trading activity, not firm values or future returns. M represents an approximation of the asset's actual traded value, not its market value.

- Shares with no high-volume days have a missing Recency value, and are excluded from RFM clusters, instead of being set to R = 0.